# CM2 vs OM2: trend comparison (winds & radiation)

**What's in this notebook**

- Compares ERA5 / JRA55-do trends against ACCESS-CM2.
- Fields: uas, vas, rsds, rlds, huss, tas.

_Split from `plots_cm2_om2_compare.ipynb`. Shared helpers in `functions.py`; outputs stripped — re-run to regenerate figures._


In [1]:
# Load Packages 

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import math
import cartopy.crs as ccrs
import calendar
from matplotlib.lines import Line2D
import os
import re
import seaborn as sns
import pandas as pd
import matplotlib.gridspec as gridspec
from scipy.stats import norm
# from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from scipy.stats import scoreatpercentile
# import xesmf as xe
from matplotlib.gridspec import GridSpec
import matplotlib

%matplotlib inline


In [2]:
import sys, os
# make this repo's functions.py importable from any subfolder
_p = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(_p, 'functions.py')):
        sys.path.insert(0, _p); break
    _p = os.path.dirname(_p)
from functions import *  # shared helper functions


In [ ]:
# Load variables produced by earlier (compute/loader) notebooks.
# Run those first; harmless if there's nothing saved yet.
load_state(globals())


# plots cm2 om2 compare

In [ ]:
# Load Packages 

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import math
import cartopy.crs as ccrs
import calendar
from matplotlib.lines import Line2D
import os
import re
import seaborn as sns
import pandas as pd
import matplotlib.gridspec as gridspec
from scipy.stats import norm
# from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from scipy.stats import scoreatpercentile
# import xesmf as xe
from matplotlib.gridspec import GridSpec
import matplotlib

%matplotlib inline


In [ ]:
cm2_trends = xr.open_dataset('/Users/isabelaconde/Desktop/data/ACCESS_CM2/ACCESS-CM2_trends_1982-2017_gr.nc')
cm2_huss_tas = xr.open_dataset('/Users/isabelaconde/Desktop/data/ACCESS_CM2/ACCESS-CM2_trends_1982-2017_gr_new.nc')

In [3]:
# base 
base_path = '/Users/isabelaconde/Desktop/data/ERA5_JRA55-do_trends/'

# ERA5 files 
era5_uas = xr.open_dataarray(base_path + 'ERA5_u10_trend_1982-2018_gr.nc').sel(lon = slice(110,300))
era5_vas = xr.open_dataarray(base_path + 'ERA5_v10_trend_1982-2018_gr.nc').sel(lon = slice(110,300))
era5_tas = xr.open_dataarray(base_path + 'ERA5_tas_trend_1982-2018_gr.nc').sel(lon = slice(110,300))
era5_huss = xr.open_dataarray(base_path + 'ERA5_huss_trend_1982-2018_gr.nc').sel(lon = slice(110,300))
era5_rsds = xr.open_dataarray(base_path + 'ERA5_rsds_trend_1982-2018_gr.nc').sel(lon = slice(110,300))
era5_rlds = xr.open_dataarray(base_path + 'ERA5_rlds_trend_1982-2018_gr.nc').sel(lon = slice(110,300))

# JRA55-do files

jra55_uas = xr.open_dataarray(base_path + 'JRA55-do_uas-trend_1982-2018_gr.nc').sel(lon = slice(110,300))
jra55_vas = xr.open_dataarray(base_path + 'JRA55-do_vas-trend_1982-2018_gr.nc').sel(lon = slice(110,300))
jra55_tas = xr.open_dataarray(base_path + 'JRA55-do_tas-trend_1982-2018_gr.nc').sel(lon = slice(110,300))
jra55_huss = xr.open_dataarray(base_path + 'JRA55-do_huss-trend_1982-2018_gr.nc').sel(lon = slice(110,300))
jra55_rsds = xr.open_dataarray(base_path + 'JRA55-do_rsds-trend_1982-2018_gr.nc').sel(lon = slice(110,300))
jra55_rlds = xr.open_dataarray(base_path + 'JRA55-do_rlds-trend_1982-2018_gr.nc').sel(lon = slice(110,300))




In [4]:

# target grid = ERA5
lat, lon = era5_vas.lat, era5_vas.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5 = era5_vas
jra55 = on_era5(jra55_vas)
cm2  = on_era5(cm2_trends.vas_trend)

das = [
    ("ERA5",                     era5*10*12),
    ("JRA55",                    jra55*10*12*30*8),
    ("ACCESS-CM2",               cm2*10*12),
    (r"JRA55 $-$ ERA5",     (jra55 - era5)*10*12),
    (r"CM2 $-$ ERA5",       (cm2  - era5)*10*12),
    (r"CM2 $-$ JRA55",      (cm2  - jra55*30*8)*10*12),
]

stack = xr.concat([d for _, d in das], dim="panel")
v = float(np.nanmax(np.abs(stack.values)))
vmin, vmax = -0.4, 0.4

proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(3, 3, figure=fig, wspace=0.03, hspace=-0.07)

axes = [fig.add_subplot(gs[r, c], projection=proj) for r in range(3) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(da.lon, da.lat, da, cmap="PuOr_r",
                       vmin=vmin, vmax=vmax, transform=pc0)
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

for ax in axes[len(das):]:
    ax.set_visible(False)

gls = []
for ax in axes[:len(das)]:
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    gls.append(gl)

for i, gl in enumerate(gls):
    if (i % 3) != 0:
        gl.left_labels = False

cax = fig.add_axes([0.92, 0.415, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'  m s$^{-1}$ (10 yr)$^{-1}$')
plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of Meridional Wind Speed', fontsize = 14, y=0.93)
plt.show()



NameError: name 'cm2_trends' is not defined

In [ ]:

# target grid = ERA5
lat, lon = era5_uas.lat, era5_uas.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5 = era5_uas
jra55 = on_era5(jra55_uas)
cm2  = on_era5(cm2_trends.uas_trend)

das = [
    ("ERA5",                     era5*10*12),
    ("JRA55",                    jra55*10*12*30*8),
    ("ACCESS-CM2",               cm2*10*12),
    (r"JRA55 $-$ ERA5",     (jra55 - era5)*10*12),
    (r"CM2 $-$ ERA5",       (cm2  - era5)*10*12),
    (r"CM2 $-$ JRA55",      (cm2  - jra55*30*8)*10*12),
]

stack = xr.concat([d for _, d in das], dim="panel")
v = float(np.nanmax(np.abs(stack.values)))
vmin, vmax = -0.4, 0.4

proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(3, 3, figure=fig, wspace=0.03, hspace=-0.07)

axes = [fig.add_subplot(gs[r, c], projection=proj) for r in range(3) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(da.lon, da.lat, da, cmap="PuOr_r",
                       vmin=vmin, vmax=vmax, transform=pc0)
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

for ax in axes[len(das):]:
    ax.set_visible(False)

gls = []
for ax in axes[:len(das)]:
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    gls.append(gl)

for i, gl in enumerate(gls):
    if (i % 3) != 0:
        gl.left_labels = False

cax = fig.add_axes([0.92, 0.415, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'  m s$^{-1}$ (10 yr)$^{-1}$')
plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of Zonal Wind Speed', fontsize = 14, y=0.93)
plt.show()



In [ ]:

lat, lon = era5_vas.lat, era5_vas.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5 = era5_vas
jra55 = on_era5(jra55_vas)

das = [
    ("ERA5",            era5 * 10 * 12),
    ("JRA55",           jra55 * 10 * 12 * 30 * 8),
    (r"ERA5 $-$ JRA55", (era5 - jra55* 30 * 8) * 10 * 12),
]

stack = xr.concat([d for _, d in das], dim="panel")
v = 0.4
vmin, vmax = -v, v  # symmetric

proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 2))
gs = GridSpec(1, 3, figure=fig, wspace=0.03)

axes = [fig.add_subplot(gs[0, c], projection=proj) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(
        da.lon, da.lat, da,
        cmap="PuOr_r", vmin=vmin, vmax=vmax,
        transform=pc0
    )
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

# gridlines: labels only on leftmost
for i, ax in enumerate(axes):
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    if i != 0:
        gl.left_labels = False

# colorbar
cax = fig.add_axes([0.92, 0.30, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'm s$^{-1}$ (10 yr)$^{-1}$')

plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of Meridional Wind Speed', fontsize=14, y=0.98)
plt.show()


In [ ]:

lat, lon = era5_vas.lat, era5_vas.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5 = era5_uas
jra55 = on_era5(jra55_uas)

das = [
    ("ERA5",            era5 * 10 * 12),
    ("JRA55",           jra55 * 10 * 12 * 30 * 8),
    (r"ERA5 $-$ JRA55", (era5 - jra55* 30 * 8) * 10 * 12),
]

stack = xr.concat([d for _, d in das], dim="panel")
v = 0.4
vmin, vmax = -v, v  # symmetric

proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 2))
gs = GridSpec(1, 3, figure=fig, wspace=0.03)

axes = [fig.add_subplot(gs[0, c], projection=proj) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(
        da.lon, da.lat, da,
        cmap="PuOr_r", vmin=vmin, vmax=vmax,
        transform=pc0
    )
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

# gridlines: labels only on leftmost
for i, ax in enumerate(axes):
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    if i != 0:
        gl.left_labels = False

# colorbar
cax = fig.add_axes([0.92, 0.30, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'm s$^{-1}$ (10 yr)$^{-1}$')

plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of Zonal Wind Speed', fontsize=14, y=0.98)
plt.show()


In [ ]:
# target grid = ERA5
lat, lon = era5_rsds.lat, era5_rsds.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5 = era5_rsds
jra55 = on_era5(jra55_rsds)
cm2  = on_era5(cm2_trends.rsds_trend)

das = [
    ("ERA5",                     era5*10*12),
    ("JRA55",                    jra55*10*12*30*8),
    ("ACCESS-CM2",               cm2*10*12),
    (r"JRA55 $-$ ERA5",     (jra55*30*8 - era5)*10*12),
    (r"CM2 $-$ ERA5",       (cm2  - era5)*10*12),
    (r"CM2 $-$ JRA55",      (cm2  - jra55*30*8)*10*12),
]



# --- common colour limits (symmetric about 0) ---
stack = xr.concat([d for _, d in das], dim="panel")
v = float(np.nanmax(np.abs(stack.values)))
vmin, vmax = -9, 9

# --- map setup (same style as your example) ---
proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(3, 3, figure=fig, wspace=0.03, hspace=-0.07)

axes = [fig.add_subplot(gs[r, c], projection=proj) for r in range(3) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(da.lon, da.lat, da, cmap="RdBu_r",
                       vmin=vmin, vmax=vmax, transform=pc0)
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

for ax in axes[len(das):]:
    ax.set_visible(False)

gls = []
for ax in axes[:len(das)]:
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    gls.append(gl)

for i, gl in enumerate(gls):
    if (i % 3) != 0:
        gl.left_labels = False

cax = fig.add_axes([0.92, 0.415, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'  W m$^{-2}$ (10 yr)$^{-1}$')
plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of Downwelling Shortwave Radiation', fontsize = 14, y=0.93)
plt.show()




In [ ]:

lat, lon = era5_rsds.lat, era5_rsds.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5  = era5_rsds
jra55 = on_era5(jra55_rsds)

das = [
    ("ERA5",            era5 * 10 * 12),
    ("JRA55",           jra55 * 10 * 12 * 30 * 8),
    (r"ERA5 $-$ JRA55", (era5 - jra55* 30 * 8) * 10 * 12),
]

# --- common colour limits (symmetric about 0) ---
stack = xr.concat([d for _, d in das], dim="panel")
v = 9
vmin, vmax = -v, v  # symmetric

# --- map setup (same style as your final example) ---
proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 2))
gs = GridSpec(1, 3, figure=fig, wspace=0.03)

axes = [fig.add_subplot(gs[0, c], projection=proj) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(
        da.lon, da.lat, da,
        cmap="RdBu_r", vmin=vmin, vmax=vmax,
        transform=pc0
    )
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

# gridlines: labels only on leftmost
for i, ax in enumerate(axes):
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    if i != 0:
        gl.left_labels = False

# colorbar
cax = fig.add_axes([0.92, 0.30, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'W m$^{-2}$ (10 yr)$^{-1}$')

plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of Downwelling Shortwave Radiation', fontsize=14, y=0.98)
plt.show()


In [ ]:

# target grid = ERA5
lat, lon = era5_rlds.lat, era5_rlds.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5 = era5_rlds
jra55 = on_era5(jra55_rlds)
cm2  = on_era5(cm2_trends.rlds_trend)

das = [
    ("ERA5",                     era5*10*12),
    ("JRA55",                    jra55*10*12*30*8),
    ("ACCESS-CM2",               cm2*10*12),
    (r"JRA55 $-$ ERA5",     (jra55*30*8 - era5)*10*12),
    (r"CM2 $-$ ERA5",       (cm2  - era5)*10*12),
    (r"CM2 $-$ JRA55",      (cm2  - jra55*30*8)*10*12),
]



# --- common colour limits (symmetric about 0) ---
stack = xr.concat([d for _, d in das], dim="panel")
v = float(np.nanmax(np.abs(stack.values)))
vmin, vmax = -3, 3

# --- map setup (same style as your example) ---
proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(3, 3, figure=fig, wspace=0.03, hspace=-0.07)

axes = [fig.add_subplot(gs[r, c], projection=proj) for r in range(3) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(da.lon, da.lat, da, cmap="RdBu_r",
                       vmin=vmin, vmax=vmax, transform=pc0)
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

for ax in axes[len(das):]:
    ax.set_visible(False)

gls = []
for ax in axes[:len(das)]:
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    gls.append(gl)

for i, gl in enumerate(gls):
    if (i % 3) != 0:
        gl.left_labels = False

cax = fig.add_axes([0.92, 0.415, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'  m s$^{-1}$ (10 yr)$^{-1}$')
plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of Downwelling Longwave Radiation', fontsize = 14, y=0.93)
plt.show()



In [ ]:

# target grid = ERA5
lat, lon = era5_rlds.lat, era5_rlds.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5  = era5_rlds
jra55 = on_era5(jra55_rlds)

das = [
    ("ERA5",            era5 * 10 * 12),
    ("JRA55",           jra55 * 10 * 12 * 30 * 8),
    (r"ERA5 $-$ JRA55", (era5 - jra55* 30 * 8) * 10 * 12),
]

# --- common colour limits (symmetric about 0) ---
stack = xr.concat([d for _, d in das], dim="panel")
v = 3
vmin, vmax = -v, v  # symmetric

# --- map setup (same style as your final 1x3 example) ---
proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 2))
gs = GridSpec(1, 3, figure=fig, wspace=0.03)

axes = [fig.add_subplot(gs[0, c], projection=proj) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(
        da.lon, da.lat, da,
        cmap="RdBu_r", vmin=vmin, vmax=vmax,
        transform=pc0
    )
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

# gridlines: labels only on leftmost
for i, ax in enumerate(axes):
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    if i != 0:
        gl.left_labels = False

# colorbar (match your placement)
cax = fig.add_axes([0.92, 0.30, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'W m$^{-2}$ (10 yr)$^{-1}$')

plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of Downwelling Longwave Radiation', fontsize=14, y=0.98)
plt.show()


In [ ]:

# target grid = ERA5
lat, lon = era5_huss.lat, era5_huss.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5 = era5_huss
jra55 = on_era5(jra55_huss)
cm2  = on_era5(cm2_huss_tas.huss_trend)

das = [
    ("ERA5",                     era5*10*12),
    ("JRA55",                    jra55*10*12*30*8),
    ("ACCESS-CM2",               cm2*10*12),
    (r"JRA55 $-$ ERA5",     (jra55*30*8 - era5)*10*12),
    (r"CM2 $-$ ERA5",       (cm2  - era5)*10*12),
    (r"CM2 $-$ JRA55",      (cm2  - jra55*30*8)*10*12),
]



# --- common colour limits (symmetric about 0) ---
stack = xr.concat([d for _, d in das], dim="panel")
v = float(np.nanmax(np.abs(stack.values)))
vmin, vmax = -0.0003, 0.0003

# --- map setup (same style as your example) ---
proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(3, 3, figure=fig, wspace=0.03, hspace=-0.07)

axes = [fig.add_subplot(gs[r, c], projection=proj) for r in range(3) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(da.lon, da.lat, da, cmap="RdYlGn",
                       vmin=vmin, vmax=vmax, transform=pc0)
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

for ax in axes[len(das):]:
    ax.set_visible(False)

gls = []
for ax in axes[:len(das)]:
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    gls.append(gl)

for i, gl in enumerate(gls):
    if (i % 3) != 0:
        gl.left_labels = False

cax = fig.add_axes([0.92, 0.415, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'  kg kg$^{-1}$ (10 yr)$^{-1}$')
plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of 2m Specific Humidity', fontsize = 14, y=0.93)
plt.show()



In [ ]:

# target grid = ERA5
lat, lon = era5_huss.lat, era5_huss.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5  = era5_huss
jra55 = on_era5(jra55_huss)

das = [
    ("ERA5",            era5 * 10 * 12),
    ("JRA55",           jra55 * 10 * 12 * 30 * 8),
    (r"ERA5 $-$ JRA55", (era5 - jra55* 30 * 8) * 10 * 12),
]

# --- common colour limits (symmetric about 0) ---
stack = xr.concat([d for _, d in das], dim="panel")
v = 0.0003
vmin, vmax = -v, v  # symmetric

# --- map setup (same style as your final 1x3 example) ---
proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 2))
gs = GridSpec(1, 3, figure=fig, wspace=0.03)

axes = [fig.add_subplot(gs[0, c], projection=proj) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(
        da.lon, da.lat, da,
        cmap="RdYlGn", vmin=vmin, vmax=vmax,
        transform=pc0
    )
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

# gridlines: labels only on leftmost
for i, ax in enumerate(axes):
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    if i != 0:
        gl.left_labels = False

# colorbar
cax = fig.add_axes([0.92, 0.30, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'kg kg$^{-1}$ (10 yr)$^{-1}$')

plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of 2m Specific Humidity', fontsize=14, y=0.98)
plt.show()


In [ ]:

# target grid = ERA5
lat, lon = era5_tas.lat, era5_tas.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5 = era5_tas*0 + 0.016/12
jra55 = on_era5(jra55_tas)
cm2  = on_era5(cm2_huss_tas.tas_trend)

das = [
    ("ERA5",                     era5*10*12),
    ("JRA55",                    jra55*10*12*30*8),
    ("ACCESS-CM2",               cm2*10*12),
    (r"JRA55 $-$ ERA5",     (jra55*30*8 - era5)*10*12),
    (r"CM2 $-$ ERA5",       (cm2  - era5)*10*12),
    (r"CM2 $-$ JRA55",      (cm2  - jra55*30*8)*10*12),
]



# --- common colour limits (symmetric about 0) ---
stack = xr.concat([d for _, d in das], dim="panel")
v = float(np.nanmax(np.abs(stack.values)))
vmin, vmax = -0.3, 0.3

# --- map setup (same style as your example) ---
proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(3, 3, figure=fig, wspace=0.03, hspace=-0.07)

axes = [fig.add_subplot(gs[r, c], projection=proj) for r in range(3) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(da.lon, da.lat, da, cmap="RdBu_r",
                       vmin=vmin, vmax=vmax, transform=pc0)
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

for ax in axes[len(das):]:
    ax.set_visible(False)

gls = []
for ax in axes[:len(das)]:
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    gls.append(gl)

for i, gl in enumerate(gls):
    if (i % 3) != 0:
        gl.left_labels = False

cax = fig.add_axes([0.92, 0.415, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'  C$^{\circ}$ (10 yr)$^{-1}$')
plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of Surface Air Temperature', fontsize = 14, y=0.93)
plt.show()



In [ ]:

# target grid = ERA5
lat, lon = era5_tas.lat, era5_tas.lon
on_era5  = lambda da: da.interp(lat=lat, lon=lon)

era5  = era5_tas
jra55 = on_era5(jra55_tas)

das = [
    ("ERA5",            era5 * 10 * 12),
    ("JRA55",           jra55 * 10 * 12 * 30 * 8),
    (r"ERA5 $-$ JRA55", (era5 - jra55* 30 * 8) * 10 * 12),
]

# --- common colour limits (symmetric about 0) ---
stack = xr.concat([d for _, d in das], dim="panel")
v = 0.3
vmin, vmax = -v, v  # symmetric

# --- map setup (same style as your final 1x3 example) ---
proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 290, -20, 20]

fig = plt.figure(figsize=(16, 2))
gs = GridSpec(1, 3, figure=fig, wspace=0.03)

axes = [fig.add_subplot(gs[0, c], projection=proj) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(
        da.lon, da.lat, da,
        cmap="RdBu_r", vmin=vmin, vmax=vmax,
        transform=pc0
    )
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

# gridlines: labels only on leftmost
for i, ax in enumerate(axes):
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    if i != 0:
        gl.left_labels = False

# colorbar
cax = fig.add_axes([0.92, 0.30, 0.012, 0.40])
fig.colorbar(ims[0], cax=cax, label=r'C$^{\circ}$ (10 yr)$^{-1}$')

plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.suptitle('Trend of Surface Air Temperature', fontsize=14, y=0.98)
plt.show()


In [ ]:
# Save this notebook's variables so later (plotting) notebooks can load them.
save_state(globals())
